In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv("metric.csv", low_memory=False)
trace_df = pd.read_csv("trace.csv", low_memory=False)
log_df = pd.read_csv("log.csv", low_memory=False)
error_logs_df = pd.read_csv("error_logs.csv", low_memory=False)

# Helper to produce compact list samples (as string) limited to n items
def sample_list_str(seq, n=50):
    seq = list(seq)
    sample = seq[:n]
    return "[" + ", ".join(map(str, sample)) + (", ..." if len(seq) > n else "") + "]"

# Summaries for metric.csv
m_rows = metric_df.shape[0]
m_ts_min = int(metric_df['timestamp'].min()) if m_rows > 0 else None
m_ts_max = int(metric_df['timestamp'].max()) if m_rows > 0 else None
m_cmdb_unique = sorted(metric_df['cmdb_id'].dropna().unique())
m_kpi_unique = sorted(metric_df['kpi_name'].dropna().unique())
m_summary = pd.DataFrame({
    'field': [
        'file', 'rows', 'timestamp_min', 'timestamp_max',
        'unique_cmdb_id_count', 'unique_cmdb_id_sample',
        'unique_kpi_count', 'unique_kpi_sample'
    ],
    'value': [
        'metric.csv', m_rows, m_ts_min, m_ts_max,
        len(m_cmdb_unique), sample_list_str(m_cmdb_unique, 50),
        len(m_kpi_unique), sample_list_str(m_kpi_unique, 50)
    ]
})

# Top 10 cmdb_id - kpi_name combos with counts and min/max timestamps
m_group = (metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)
           .agg(count=('timestamp', 'size'),
                ts_min=('timestamp', 'min'),
                ts_max=('timestamp', 'max'))
           .reset_index()
           .sort_values('count', ascending=False)
           .head(10)
           .reset_index(drop=True))

# Summaries for trace.csv
t_rows = trace_df.shape[0]
t_ts_min = int(trace_df['timestamp'].min()) if t_rows > 0 else None
t_ts_max = int(trace_df['timestamp'].max()) if t_rows > 0 else None
t_cmdb_unique = sorted(trace_df['cmdb_id'].dropna().unique())
t_trace_unique = sorted(trace_df['trace_name'].dropna().unique())
t_summary = pd.DataFrame({
    'field': [
        'file', 'rows', 'timestamp_min', 'timestamp_max',
        'unique_cmdb_id_count', 'unique_cmdb_id_sample',
        'unique_trace_count', 'unique_trace_sample'
    ],
    'value': [
        'trace.csv', t_rows, t_ts_min, t_ts_max,
        len(t_cmdb_unique), sample_list_str(t_cmdb_unique, 50),
        len(t_trace_unique), sample_list_str(t_trace_unique, 50)
    ]
})

# Top 10 cmdb_id - trace_name combos with counts and min/max timestamps
t_group = (trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)
           .agg(count=('timestamp', 'size'),
                ts_min=('timestamp', 'min'),
                ts_max=('timestamp', 'max'))
           .reset_index()
           .sort_values('count', ascending=False)
           .head(10)
           .reset_index(drop=True))

# Summaries for log.csv
l_rows = log_df.shape[0]
l_ts_min = int(log_df['timestamp'].min()) if l_rows > 0 else None
l_ts_max = int(log_df['timestamp'].max()) if l_rows > 0 else None
l_cmdb_unique = sorted(log_df['cmdb_id'].dropna().unique())
l_logname_unique = sorted(log_df['log_name'].dropna().unique())
l_summary = pd.DataFrame({
    'field': [
        'file', 'rows', 'timestamp_min', 'timestamp_max',
        'unique_cmdb_id_count', 'unique_cmdb_id_sample',
        'unique_logname_count', 'unique_logname_sample'
    ],
    'value': [
        'log.csv', l_rows, l_ts_min, l_ts_max,
        len(l_cmdb_unique), sample_list_str(l_cmdb_unique, 50),
        len(l_logname_unique), sample_list_str(l_logname_unique, 50)
    ]
})

# Summaries for error_logs.csv
e_rows = error_logs_df.shape[0]
e_ts_min = int(error_logs_df['timestamp'].min()) if e_rows > 0 else None
e_ts_max = int(error_logs_df['timestamp'].max()) if e_rows > 0 else None
e_cmdb_unique = sorted(error_logs_df['cmdb_id'].dropna().unique())
# count non-empty message rows
if 'message' in error_logs_df.columns:
    non_empty_mask = error_logs_df['message'].fillna("").astype(str).str.strip() != ""
    e_non_empty_count = int(non_empty_mask.sum())
    # top up to 10 cmdb_id by non-empty message counts
    e_top_cmdb = (error_logs_df[non_empty_mask]
                  .groupby('cmdb_id', dropna=False)
                  .size()
                  .reset_index(name='non_empty_message_count')
                  .sort_values('non_empty_message_count', ascending=False)
                  .head(10)
                  .reset_index(drop=True))
else:
    e_non_empty_count = 0
    e_top_cmdb = pd.DataFrame(columns=['cmdb_id', 'non_empty_message_count'])

e_summary = pd.DataFrame({
    'field': [
        'file', 'rows', 'timestamp_min', 'timestamp_max',
        'unique_cmdb_id_count', 'unique_cmdb_id_sample',
        'non_empty_message_count', 'top_cmdb_sample_count_rows'
    ],
    'value': [
        'error_logs.csv', e_rows, e_ts_min, e_ts_max,
        len(e_cmdb_unique), sample_list_str(e_cmdb_unique, 10),
        e_non_empty_count,
        f"top {min(10, len(e_top_cmdb))} by non-empty messages"
    ]
})

# Final compact outputs (limited in size)
m_summary, m_group, t_summary, t_group, l_summary, e_summary, e_top_cmdb

```
Out[1]:
```
```python
# Summarize the previously computed inventory results in plain English.
# This is a concise human-readable summary built from the existing variables:
# m_summary, m_group, t_summary, t_group, l_summary, e_summary, e_top_cmdb

def get_field(df, field):
    return df.loc[df['field'] == field, 'value'].iloc[0]

# Metric summary fields
m_rows = get_field(m_summary, 'rows')
m_ts_min = get_field(m_summary, 'timestamp_min')
m_ts_max = get_field(m_summary, 'timestamp_max')
m_cmdb_count = get_field(m_summary, 'unique_cmdb_id_count')
m_cmdb_sample = get_field(m_summary, 'unique_cmdb_id_sample')
m_kpi_count = get_field(m_summary, 'unique_kpi_count')

# Trace summary fields
t_rows = get_field(t_summary, 'rows')
t_ts_min = get_field(t_summary, 'timestamp_min')
t_ts_max = get_field(t_summary, 'timestamp_max')
t_cmdb_count = get_field(t_summary, 'unique_cmdb_id_count')
t_cmdb_sample = get_field(t_summary, 'unique_cmdb_id_sample')
t_trace_count = get_field(t_summary, 'unique_trace_count')

# Log summary fields
l_rows = get_field(l_summary, 'rows')
l_ts_min = get_field(l_summary, 'timestamp_min')
l_ts_max = get_field(l_summary, 'timestamp_max')
l_cmdb_count = get_field(l_summary, 'unique_cmdb_id_count')
l_cmdb_sample = get_field(l_summary, 'unique_cmdb_id_sample')
l_logname_count = get_field(l_summary, 'unique_logname_count')

# Error logs summary fields
e_rows = get_field(e_summary, 'rows')
e_ts_min = get_field(e_summary, 'timestamp_min')
e_ts_max = get_field(e_summary, 'timestamp_max')
e_cmdb_count = get_field(e_summary, 'unique_cmdb_id_count')
e_non_empty = get_field(e_summary, 'non_empty_message_count')

# Top combos (compact)
def combos_to_str(df, name_col):
    items = []
    for _, row in df.iterrows():
        cmdb = row['cmdb_id']
        name = row[name_col]
        cnt = int(row['count'])
        items.append(f"{cmdb} / {name} ({cnt})")
    return ", ".join(items)

metric_top = combos_to_str(m_group, 'kpi_name')
trace_top = combos_to_str(t_group, 'trace_name')

# Error logs top cmdbs
e_top_list = []
for _, r in e_top_cmdb.iterrows():
    e_top_list.append(f"{r['cmdb_id']} ({int(r['non_empty_message_count'])})")
e_top_str = ", ".join(e_top_list)

summary = (
    "Inventory summary:\n\n"
    "metric.csv:\n"
    f"- Rows: {m_rows}\n"
    f"- Timestamp range: {m_ts_min} to {m_ts_max}\n"
    f"- Unique cmdb_id count: {m_cmdb_count}; sample: {m_cmdb_sample}\n"
    f"- Unique KPI count: {m_kpi_count}\n"
    f"- Top 10 cmdb_id - KPI combos (cmdb / kpi (count)): {metric_top}\n\n"
    "trace.csv:\n"
    f"- Rows: {t_rows}\n"
    f"- Timestamp range: {t_ts_min} to {t_ts_max}\n"
    f"- Unique cmdb_id count: {t_cmdb_count}; sample: {t_cmdb_sample}\n"
    f"- Unique trace feature count: {t_trace_count}\n"
    f"- Top 10 cmdb_id - trace combos (cmdb / trace (count)): {trace_top}\n\n"
    "log.csv:\n"
    f"- Rows: {l_rows}\n"
    f"- Timestamp range: {l_ts_min} to {l_ts_max}\n"
    f"- Unique cmdb_id count: {l_cmdb_count}; sample: {l_cmdb_sample}\n"
    f"- Unique log_name count: {l_logname_count}\n\n"
    "error_logs.csv:\n"
    f"- Rows: {e_rows}\n"
    f"- Timestamp range: {e_ts_min} to {e_ts_max}\n"
    f"- Unique cmdb_id count: {e_cmdb_count}\n"
    f"- Non-empty message rows: {e_non_empty}\n"
    f"- Top cmdb_id by non-empty messages (cmdb (count)): {e_top_str}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   field                                              value
0                   file                                         metric.csv
1                   rows                                             152118
2          timestamp_min                                         1647865800
3          timestamp_max                                         1647867540
4   unique_cmdb_id_count                                                 59
5  unique_cmdb_id_sample  [adservice, adservice-0, adservice-1, adservic...
6       unique_kpi_count                                               1292
7      unique_kpi_sample  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,               cmdb_id                                           kpi_name  count      ts_min      ts_max
0  shippingservice2-0  container.node-5.container_network_receive_MB....     30  1647865800  1647867540
1  shippingservice2-0  container.node-5.container_network_receive_err...     30  1647865800  1647867540
2  shippingservice2-0  container.node-5.container_network_receive_pac...     30  1647865800  1647867540
3  shippingservice2-0  container.node-5.container_network_receive_pac...     30  1647865800  1647867540
4  shippingservice2-0  container.node-5.container_network_transmit_MB...     30  1647865800  1647867540
5  shippingservice2-0  container.node-5.container_network_transmit_er...     30  1647865800  1647867540
6  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  1647865800  1647867540
7  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  1647865800  1647867540
8  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  1647865800  1647867540
9  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  1647865800  1647867540,                    field                                              value
0                   file                                          trace.csv
1                   rows                                              35608
2          timestamp_min                                         1647865800
3          timestamp_max                                         1647867540
4   unique_cmdb_id_count                                                 40
5  unique_cmdb_id_sample  [adservice-0, adservice-1, adservice-2, adserv...
6     unique_trace_count                                                200
7    unique_trace_sample  [trace.from_checkoutservice-0.duration_mean, t...,               cmdb_id                           trace_name  count      ts_min      ts_max
0  shippingservice2-0     trace.from_frontend2-0.row_count     30  1647865800  1647867540
1         adservice-0  trace.from_frontend-0.duration_mean     30  1647865800  1647867540
2         adservice-0   trace.from_frontend-0.duration_p95     30  1647865800  1647867540
3         adservice-0     trace.from_frontend-0.error_rate     30  1647865800  1647867540
4         adservice-0      trace.from_frontend-0.row_count     30  1647865800  1647867540
5         adservice-0  trace.from_frontend-1.duration_mean     30  1647865800  1647867540
6         adservice-0   trace.from_frontend-1.duration_p95     30  1647865800  1647867540
7         adservice-0     trace.from_frontend-1.error_rate     30  1647865800  1647867540
8         adservice-0      trace.from_frontend-1.row_count     30  1647865800  1647867540
9         adservice-0  trace.from_frontend-2.duration_mean     30  1647865800  1647867540,                    field                                              value
0                   file                                            log.csv
1                   rows                                               1814
2          timestamp_min                                         1647865800
3          timestamp_max                                         1647867540
4   unique_cmdb_id_count                                                 31
5  unique_cmdb_id_sample  [adservice-0, adservice-1, adservice-2, cartse...
6   unique_logname_count                                                  2
7  unique_logname_sample                   [log.error_count, log.row_count],                         field                                              value
0                        file                                     error_logs.csv
1                        rows                                               1379
2               timestamp_min                                         1647865801
3               timestamp_max                                         1647867599
4        unique_cmdb_id_count                                                  5
5       unique_cmdb_id_sample  [checkoutservice-0, checkoutservice-2, fronten...
6     non_empty_message_count                                               1379
7  top_cmdb_sample_count_rows                        top 5 by non-empty messages,              cmdb_id  non_empty_message_count
0         frontend-0                      546
1         frontend-2                      412
2         frontend-1                      406
3  checkoutservice-2                        8
4  checkoutservice-0                        7)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse metric_df loaded in kernel previously.
# Ensure numeric types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df = metric_df.dropna(subset=['value'])  # drop rows with non-numeric values

# 1) Build allowed root-cause candidate set (from domain knowledge)
nodes = [f"node-{i}" for i in range(1, 7)]

services = [
    "frontend", "shippingservice", "checkoutservice", "currencyservice", "adservice",
    "emailservice", "cartservice", "productcatalogservice", "recommendationservice",
    "paymentservice"
]

# pods listed in domain knowledge (include "-0","-1","-2" and "2-0" variants where given)
pods = []
for svc in services:
    pods.extend([f"{svc}-0", f"{svc}-1", f"{svc}-2", f"{svc}2-0"])
# Deduplicate and include examples that were in metric samples (some may not exist)
allowed_set = set(nodes + services + pods)

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire timeseries (before any time filtering)
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed_set)].copy()

# It's possible no rows exist for some allowed cmdb_id; proceed with existing ones.
p95_series = metric_allowed.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value': 'p95'})

# 2) Filter metric.csv to the incident time window Mar 21, 2022 20:30 to 21:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 21, 20, 30, 0))
end_dt = tz.localize(datetime(2022, 3, 21, 21, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

window_df = metric_allowed[(metric_allowed['timestamp'] >= start_ts) & (metric_allowed['timestamp'] <= end_ts)].copy()

# Merge p95 into window rows
window_df = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Compute is_anom mask: value >= p95 (handle missing p95 by treating as False)
window_df['is_anom'] = False
mask_p95_valid = window_df['p95'].notna()
window_df.loc[mask_p95_valid, 'is_anom'] = window_df.loc[mask_p95_valid, 'value'] >= window_df.loc[mask_p95_valid, 'p95']

# Aggregations per (cmdb_id, kpi_name)
agg_total = window_df.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    total_points_in_window=('timestamp', 'size'),
    max_value_in_window=('value', 'max')
).reset_index()

agg_anom_counts = window_df[window_df['is_anom']].groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    anomaly_count=('is_anom', 'sum'),
    earliest_anomaly_timestamp=('timestamp', 'min')
).reset_index()

# Merge aggregates and p95
summary_df = agg_total.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
summary_df = summary_df.merge(agg_anom_counts, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for combos with no anomalies
summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
summary_df['earliest_anomaly_timestamp'] = summary_df['earliest_anomaly_timestamp'].where(summary_df['earliest_anomaly_timestamp'].notna(), np.nan)

# Compute severity_ratio = max_value_in_window / p95 (handle p95==0 or NaN)
summary_df['severity_ratio'] = np.where(
    (summary_df['p95'].notna()) & (summary_df['p95'] != 0),
    summary_df['max_value_in_window'] / summary_df['p95'],
    np.nan
)

# 3) Top 20 combos ranked by anomaly_count (desc)
top20 = (summary_df
         .sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False])
         .head(20)
         .reset_index(drop=True)
        )

# Select and order required columns compactly
top20_display = top20[[
    'cmdb_id',
    'kpi_name',
    'p95',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'severity_ratio',
    'total_points_in_window'
]].copy()

# Count distinct combos with anomaly_count > 0
distinct_anomalous_combos = int((summary_df['anomaly_count'] > 0).sum())

# Keep output compact: show top20_display and the scalar count
top20_display, distinct_anomalous_combos

```
Out[2]:
```
```python
# Build a plain-English summary string from the kernel results (top20_display, distinct_anomalous_combos).
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Use existing kernel variables: top20_display (DataFrame) and distinct_anomalous_combos (int)
# Prepare top N rows to include in summary (limit to 10 for compactness)
top_n = 10
top_rows = top20_display.head(top_n)

lines = []
lines.append("Anomaly analysis summary (metric.csv) — incident window: 2022-03-21 20:30 to 21:00 (UTC+8)")
lines.append("")
lines.append(f"- Number of distinct (cmdb_id, kpi_name) combos with anomaly_count > 0: {distinct_anomalous_combos}")
lines.append("")
lines.append(f"- Top {min(top_n, len(top_rows))} (cmdb_id, kpi_name) ranked by anomaly_count:")
for i, row in top_rows.reset_index(drop=True).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['p95']
    anom = int(row['anomaly_count'])
    earliest_ts = row['earliest_anomaly_timestamp']
    if pd.isna(earliest_ts):
        earliest_str = "N/A"
    else:
        earliest_str = datetime.fromtimestamp(int(earliest_ts), tz).strftime("%Y-03-21 %H:%M:%S %Z")
        # Note: keep date formatting consistent; timestamp values are within the incident window
    max_val = row['max_value_in_window']
    sev = row['severity_ratio']
    total_pts = int(row['total_points_in_window'])
    lines.append(f"{i+1}) {cmdb} / {kpi} — anomalies: {anom}, p95: {p95}, earliest: {earliest_str}, max: {max_val}, severity_ratio: {sev:.3f}, total_points: {total_pts}")

summary_text = "\n".join(lines)

# Display the summary string (compact)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name     p95  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0                     node-5                             system.udp.connect.num     9.0             30                  1647865800            11.000000        1.222222                      30
1    productcatalogservice-0                 container.node-5.container_sockets     4.0             30                  1647865800             4.500000        1.125000                      30
2   productcatalogservice2-0                 container.node-6.container_sockets     3.0             30                  1647865800             3.333333        1.111111                      30
3                     node-2                             system.udp.connect.num    10.0             30                  1647865800            11.000000        1.100000                      30
4    productcatalogservice-0        container.node-5.container_file_descriptors    10.0             30                  1647865800            10.500000        1.050000                      30
5   productcatalogservice2-0        container.node-6.container_file_descriptors     9.0             30                  1647865800             9.333333        1.037037                      30
6                  adservice                 runtime.jvm_threads_state.RUNNABLE     5.0             30                  1647865800             5.083333        1.016667                      30
7                  adservice                                        app.grpc.rr   100.0             30                  1647865800           100.000000        1.000000                      30
8                  adservice                                     app.http.count    24.0             30                  1647865800            24.000000        1.000000                      30
9                  adservice                                        app.http.rr   100.0             30                  1647865800           100.000000        1.000000                      30
10                 adservice                                        app.http.sr   100.0             30                  1647865800           100.000000        1.000000                      30
11                 adservice    runtime.java_lang_ClassLoading_LoadedClassCount  5175.0             30                  1647865800          5175.000000        1.000000                      30
12                 adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...  5201.0             30                  1647865800          5201.000000        1.000000                      30
13                 adservice  runtime.java_lang_ClassLoading_UnloadedClassCount    26.0             30                  1647865800            26.000000        1.000000                      30
14                 adservice  runtime.java_lang_Compilation_CompilationTimeM...     1.0             30                  1647865800             1.000000        1.000000                      30
15                 adservice  runtime.java_lang_GarbageCollector_CollectionC...     5.0             30                  1647865800             5.000000        1.000000                      30
16                 adservice  runtime.java_lang_GarbageCollector_CollectionT...   145.0             30                  1647865800           145.000000        1.000000                      30
17                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     1.0             30                  1647865800             1.000000        1.000000                      30
18                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     1.0             30                  1647865800             1.000000        1.000000                      30
19                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    42.0             30                  1647865800            42.000000        1.000000                      30, 4731)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse summary_df computed earlier which contains:
# ['cmdb_id','kpi_name','p95','anomaly_count','earliest_anomaly_timestamp',
#  'max_value_in_window','severity_ratio','total_points_in_window', ...]
# Ensure summary_df exists
assert 'summary_df' in globals(), "summary_df not found in kernel."

# 1) Top 20 (cmdb_id, kpi_name) combos with anomaly_count > 0 ranked by severity_ratio desc
anom_df = summary_df[summary_df['anomaly_count'] > 0].copy()

# Sort by severity_ratio descending, keep NaNs last
top20_by_severity = (anom_df
                     .sort_values(by='severity_ratio', ascending=False, na_position='last')
                     .head(20)
                     [['cmdb_id',
                       'kpi_name',
                       'p95',
                       'anomaly_count',
                       'earliest_anomaly_timestamp',
                       'max_value_in_window',
                       'severity_ratio',
                       'total_points_in_window']]
                     .reset_index(drop=True)
                    )

# 2) Aggregate counts by cmdb_id type: node, pod, service
nodes = {f"node-{i}" for i in range(1, 7)}

def classify_cmdb(cmdb):
    if cmdb in nodes:
        return 'node'
    if '-' in str(cmdb):
        return 'pod'
    return 'service'

anom_df['cmdb_type'] = anom_df['cmdb_id'].apply(classify_cmdb)

node_count = int((anom_df['cmdb_type'] == 'node').sum())
pod_count = int((anom_df['cmdb_type'] == 'pod').sum())
service_count = int((anom_df['cmdb_type'] == 'service').sum())

counts_df = pd.DataFrame({
    'type': ['node', 'pod', 'service'],
    'anomalous_combo_count': [node_count, pod_count, service_count]
})

# 3) Earliest anomaly timestamp among all anomalous combos and top 5 cmdb_ids by distinct anomalous KPIs
# earliest (minimum) earliest_anomaly_timestamp (numeric)
# Filter non-null earliest_anomaly_timestamp before min
valid_earliest = anom_df['earliest_anomaly_timestamp'].dropna()
if not valid_earliest.empty:
    earliest_ts = int(valid_earliest.min())
    # format to UTC+8 (Asia/Shanghai)
    tz = pytz.timezone('Asia/Shanghai')
    earliest_ts_fmt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz)
else:
    earliest_ts = np.nan
    earliest_ts_fmt = pd.NaT

earliest_ts_df = pd.DataFrame({
    'earliest_anomaly_timestamp_epoch_s': [earliest_ts],
    'earliest_anomaly_timestamp_UTC+8': [earliest_ts_fmt]
})

# Top 5 cmdb_ids with highest number of distinct anomalous KPIs
top5_cmdb_kpi_counts = (anom_df.groupby('cmdb_id', dropna=False)['kpi_name']
                        .nunique()
                        .reset_index(name='distinct_anomalous_kpi_count')
                        .sort_values('distinct_anomalous_kpi_count', ascending=False)
                        .head(5)
                        .reset_index(drop=True)
                       )

# Final compact outputs
top20_by_severity, counts_df, earliest_ts_df, top5_cmdb_kpi_counts

```
Out[3]:
```
Summary of anomalies (ranked by severity_ratio, top 20):

1) frontend-1 / mesh.source.frontend.checkoutservice.istio_request...  
   - p95=0.173787, anomaly_count=2, earliest=1647866160 (2022-03-21 20:36:00 UTC+8), max=19.58675, severity_ratio=112.705, total_points=30

2) adservice2-0 / container.node-6.container_cpu_cfs_throttled_seconds_total  
   - p95=0.070435, anomaly_count=2, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=2.867174, severity_ratio=40.707, total_points=30

3) adservice2-0 / container.node-6.container_cpu_cfs_throttled_percent  
   - p95=1.0, anomaly_count=3, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=39.0, severity_ratio=39.000, total_points=30

4) emailservice-0 / mesh.destination.checkoutservice.emailservice...  
   - p95=0.004672, anomaly_count=2, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=0.12995, severity_ratio=27.812, total_points=23

5) paymentservice-0 / container.node-5.container_network_receive_MB...  
   - p95=0.024972, anomaly_count=2, earliest=1647865860 (2022-03-21 20:31:00 UTC+8), max=0.549932, severity_ratio=22.022, total_points=30

6) emailservice-2 / container.node-5.container_network_receive_MB...  
   - p95=0.027932, anomaly_count=2, earliest=1647866340 (2022-03-21 20:39:00 UTC+8), max=0.552119, severity_ratio=19.766, total_points=30

7) emailservice2-0 / container.node-5.container_network_receive_MB...  
   - p95=0.031387, anomaly_count=2, earliest=1647866280 (2022-03-21 20:38:00 UTC+8), max=0.555544, severity_ratio=17.700, total_points=30

8) shippingservice-0 / container.node-5.container_network_receive_MB...  
   - p95=0.042714, anomaly_count=2, earliest=1647866340 (2022-03-21 20:39:00 UTC+8), max=0.56913, severity_ratio=13.324, total_points=30

9) checkoutservice-0 / container.node-5.container_network_receive_MB...  
   - p95=0.055555, anomaly_count=2, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=0.574874, severity_ratio=10.348, total_points=30

10) checkoutservice-2 / container.node-5.container_network_receive_MB...  
    - p95=0.057544, anomaly_count=2, earliest=1647866160 (2022-03-21 20:36:00 UTC+8), max=0.57146, severity_ratio=9.931, total_points=30

11) adservice-2 / container.node-5.container_cpu_cfs_throttled_seconds_total  
    - p95=0.007566, anomaly_count=2, earliest=1647865800 (2022-03-21 20:30:00 UTC+8), max=0.070975, severity_ratio=9.381, total_points=30

12) adservice-1 / container.node-5.container_network_receive_MB...  
    - p95=0.067105, anomaly_count=2, earliest=1647865860 (2022-03-21 20:31:00 UTC+8), max=0.608926, severity_ratio=9.074, total_points=30

13) adservice-0 / container.node-5.container_network_receive_MB...  
    - p95=0.066126, anomaly_count=2, earliest=1647865920 (2022-03-21 20:32:00 UTC+8), max=0.586749, severity_ratio=8.873, total_points=30

14) node-2 / system.io.avg_q_sz  
    - p95=0.03, anomaly_count=3, earliest=1647867060 (2022-03-21 20:51:00 UTC+8), max=0.25, severity_ratio=8.333, total_points=30

15) cartservice-2 / container.node-5.container_network_receive_MB...  
    - p95=0.076388, anomaly_count=2, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=0.593606, severity_ratio=7.771, total_points=30

16) cartservice-0 / container.node-5.container_network_receive_MB...  
    - p95=0.080728, anomaly_count=2, earliest=1647866220 (2022-03-21 20:37:00 UTC+8), max=0.585325, severity_ratio=7.251, total_points=30

17) cartservice-1 / container.node-5.container_network_receive_MB...  
    - p95=0.083038, anomaly_count=2, earliest=1647866340 (2022-03-21 20:39:00 UTC+8), max=0.600687, severity_ratio=7.234, total_points=30

18) frontend-1 / mesh.destination.unknown.frontend.istio_request...  
    - p95=3.256687, anomaly_count=2, earliest=1647867000 (2022-03-21 20:50:00 UTC+8), max=22.284, severity_ratio=6.843, total_points=30

19) node-5 / system.io.avg_q_sz  
    - p95=0.071, anomaly_count=2, earliest=1647866040 (2022-03-21 20:34:00 UTC+8), max=0.47, severity_ratio=6.620, total_points=30

20) adservice-2 / container.node-5.container_cpu_cfs_throttled_percent  
    - p95=0.5, anomaly_count=3, earliest=1647865800 (2022-03-21 20:30:00 UTC+8), max=3.0, severity_ratio=6.000, total_points=30

Aggregate counts (distinct anomalous (cmdb_id, kpi_name) combos where anomaly_count > 0):
- node (node-1 .. node-6): 350
- pod (cmdb_id contains a dash and is not a node): 4,004
- service (cmdb_id contains no dash): 377

Earliest anomaly observed among all anomalous combos:
- epoch 1647865800 → 2022-03-21 20:30:00 (UTC+8)

Top 5 cmdb_ids by number of distinct anomalous KPIs:
1) adservice — 341 distinct anomalous KPIs  
2) frontend2-0 — 184  
3) frontend-1 — 180  
4) frontend-2 — 180  
5) frontend-0 — 180

(Notes: timestamps are shown in UTC+8 as requested; lists are limited to the requested maxima.)

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name       p95  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window
0          frontend-1  mesh.source.frontend.checkoutservice.istio_req...  0.173787              2                  1647866160            19.586750      112.705172                      30
1        adservice2-0  container.node-6.container_cpu_cfs_throttled_s...  0.070435              2                  1647866040             2.867174       40.706767                      30
2        adservice2-0  container.node-6.container_cpu_cfs_throttled_p...  1.000000              3                  1647866040            39.000000       39.000000                      30
3      emailservice-0  mesh.destination.checkoutservice.emailservice....  0.004672              2                  1647866040             0.129950       27.811664                      23
4    paymentservice-0  container.node-5.container_network_receive_MB....  0.024972              2                  1647865860             0.549932       22.021934                      30
5      emailservice-2  container.node-5.container_network_receive_MB....  0.027932              2                  1647866340             0.552119       19.766195                      30
6     emailservice2-0  container.node-5.container_network_receive_MB....  0.031387              2                  1647866280             0.555544       17.699907                      30
7   shippingservice-0  container.node-5.container_network_receive_MB....  0.042714              2                  1647866340             0.569130       13.324228                      30
8   checkoutservice-0  container.node-5.container_network_receive_MB....  0.055555              2                  1647866040             0.574874       10.347894                      30
9   checkoutservice-2  container.node-5.container_network_receive_MB....  0.057544              2                  1647866160             0.571460        9.930841                      30
10        adservice-2  container.node-5.container_cpu_cfs_throttled_s...  0.007566              2                  1647865800             0.070975        9.381289                      30
11        adservice-1  container.node-5.container_network_receive_MB....  0.067105              2                  1647865860             0.608926        9.074282                      30
12        adservice-0  container.node-5.container_network_receive_MB....  0.066126              2                  1647865920             0.586749        8.873160                      30
13             node-2                                 system.io.avg_q_sz  0.030000              3                  1647867060             0.250000        8.333333                      30
14      cartservice-2  container.node-5.container_network_receive_MB....  0.076388              2                  1647866040             0.593606        7.770970                      30
15      cartservice-0  container.node-5.container_network_receive_MB....  0.080728              2                  1647866220             0.585325        7.250617                      30
16      cartservice-1  container.node-5.container_network_receive_MB....  0.083038              2                  1647866340             0.600687        7.233909                      30
17         frontend-1  mesh.destination.unknown.frontend.istio_reques...  3.256687              2                  1647867000            22.284000        6.842536                      30
18             node-5                                 system.io.avg_q_sz  0.071000              2                  1647866040             0.470000        6.619718                      30
19        adservice-2  container.node-5.container_cpu_cfs_throttled_p...  0.500000              3                  1647865800             3.000000        6.000000                      30,       type  anomalous_combo_count
0     node                    350
1      pod                   4004
2  service                    377,    earliest_anomaly_timestamp_epoch_s earliest_anomaly_timestamp_UTC+8
0                          1647865800        2022-03-21 20:30:00+08:00,        cmdb_id  distinct_anomalous_kpi_count
0    adservice                           341
1  frontend2-0                           184
2   frontend-1                           180
3   frontend-2                           180
4   frontend-0                           180)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse error_logs_df from kernel
assert 'error_logs_df' in globals(), "error_logs_df not found in kernel."

# Focused frontends
frontends = ['frontend-0', 'frontend-1', 'frontend-2']

# Ensure message column exists and compute non-empty mask
if 'message' in error_logs_df.columns:
    error_logs_df['message_str'] = error_logs_df['message'].fillna("").astype(str)
    error_logs_df['message_nonempty'] = error_logs_df['message_str'].str.strip() != ""
else:
    error_logs_df['message_str'] = ""
    error_logs_df['message_nonempty'] = False

# Filter rows for the three frontends
frontend_logs = error_logs_df[error_logs_df['cmdb_id'].isin(frontends)].copy()

# 1) For each frontend: total non-empty message count, earliest ts, latest ts
per_frontend = (frontend_logs[frontend_logs['message_nonempty']]
                .groupby('cmdb_id', dropna=False)
                .agg(
                    non_empty_message_count=('message_nonempty', 'sum'),
                    earliest_timestamp_epoch_s=('timestamp', 'min'),
                    latest_timestamp_epoch_s=('timestamp', 'max')
                )
                .reset_index()
               )

# Add UTC+8 formatted times (Asia/Shanghai) using .dt accessor safely
tz = pytz.timezone('Asia/Shanghai')

def fmt_epoch_series(s):
    if s.dropna().empty:
        return pd.Series([pd.NaT] * len(s), index=s.index)
    return pd.to_datetime(s.astype('Int64'), unit='s', utc=True).dt.tz_convert(tz)

# Create formatted columns safely (handle empties)
if not per_frontend.empty:
    per_frontend['earliest_timestamp_UTC+8'] = fmt_epoch_series(per_frontend['earliest_timestamp_epoch_s'])
    per_frontend['latest_timestamp_UTC+8'] = fmt_epoch_series(per_frontend['latest_timestamp_epoch_s'])
else:
    per_frontend['earliest_timestamp_UTC+8'] = pd.Series(dtype='datetime64[ns]')
    per_frontend['latest_timestamp_UTC+8'] = pd.Series(dtype='datetime64[ns]')

# Keep columns compact and ordered
per_frontend_summary = per_frontend[[
    'cmdb_id',
    'non_empty_message_count',
    'earliest_timestamp_epoch_s',
    'earliest_timestamp_UTC+8',
    'latest_timestamp_epoch_s',
    'latest_timestamp_UTC+8'
]].copy()

# Ensure all three frontends appear (include zeros if missing)
missing = [f for f in frontends if f not in per_frontend_summary['cmdb_id'].values]
if missing:
    for m in missing:
        per_frontend_summary = pd.concat([per_frontend_summary,
                                          pd.DataFrame([{
                                              'cmdb_id': m,
                                              'non_empty_message_count': 0,
                                              'earliest_timestamp_epoch_s': pd.NA,
                                              'earliest_timestamp_UTC+8': pd.NaT,
                                              'latest_timestamp_epoch_s': pd.NA,
                                              'latest_timestamp_UTC+8': pd.NaT
                                          }])], ignore_index=True)

per_frontend_summary = per_frontend_summary.sort_values('cmdb_id').reset_index(drop=True)

# 2) Per-minute message counts across all three in incident window and top 10 minutes
start_dt = tz.localize(datetime(2022, 3, 21, 20, 30, 0))
end_dt = tz.localize(datetime(2022, 3, 21, 21, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

window_frontend = frontend_logs[
    (frontend_logs['timestamp'] >= start_ts) & (frontend_logs['timestamp'] <= end_ts) & (frontend_logs['message_nonempty'])
].copy()

# Group by timestamp (per-minute) across all three frontends
per_minute = (window_frontend
              .groupby('timestamp', dropna=False)
              .size()
              .reset_index(name='message_count')
             )

# Add formatted UTC+8 time using safe dt accessor
if not per_minute.empty:
    per_minute['timestamp_UTC+8'] = pd.to_datetime(per_minute['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
else:
    per_minute['timestamp_UTC+8'] = pd.Series(dtype='datetime64[ns]')

top10_minutes = per_minute.sort_values('message_count', ascending=False).head(10).reset_index(drop=True)
top10_minutes = top10_minutes[['timestamp', 'timestamp_UTC+8', 'message_count']]

# 3) Single earliest non-empty message timestamp among these frontends (scalar)
if not frontend_logs[frontend_logs['message_nonempty']].empty:
    earliest_overall_epoch = int(frontend_logs[frontend_logs['message_nonempty']]['timestamp'].min())
    earliest_overall_UTC8 = pd.to_datetime(earliest_overall_epoch, unit='s', utc=True).tz_convert(tz)
else:
    earliest_overall_epoch = pd.NA
    earliest_overall_UTC8 = pd.NaT

earliest_overall = pd.Series({
    'earliest_non_empty_timestamp_epoch_s': earliest_overall_epoch,
    'earliest_non_empty_timestamp_UTC+8': earliest_overall_UTC8
})

# Return compact structured outputs
per_frontend_summary, top10_minutes, earliest_overall

```
Out[4]:
```
```python
# Build a concise plain-English summary from the previously computed kernel tables:
# per_frontend_summary, top10_minutes, earliest_overall

lines = []
lines.append("Frontend error_logs summary (frontend-0, frontend-1, frontend-2):")
lines.append("")
# 1) Per-frontend totals and time ranges
lines.append("1) Per-frontend non-empty message counts and time ranges:")
for _, r in per_frontend_summary.sort_values('cmdb_id').iterrows():
    cmdb = r['cmdb_id']
    cnt = int(r['non_empty_message_count'])
    est_epoch = r['earliest_timestamp_epoch_s']
    let_epoch = r['latest_timestamp_epoch_s']
    est_str = str(r['earliest_timestamp_UTC+8']) if pd.notna(r['earliest_timestamp_UTC+8']) else "N/A"
    let_str = str(r['latest_timestamp_UTC+8']) if pd.notna(r['latest_timestamp_UTC+8']) else "N/A"
    lines.append(f"- {cmdb}: {cnt} messages; earliest={est_epoch} ({est_str}); latest={let_epoch} ({let_str})")

lines.append("")
# 2) Top 10 per-minute message counts in the incident window
lines.append("2) Top 10 minutes in incident window (2022-03-21 20:30–21:00 UTC+8) by message count across the three frontends:")
if top10_minutes.empty:
    lines.append("- No messages in the window.")
else:
    for _, r in top10_minutes.iterrows():
        ts = int(r['timestamp'])
        ts_str = str(r['timestamp_UTC+8'])
        cnt = int(r['message_count'])
        lines.append(f"- {ts} ({ts_str}): {cnt}")

lines.append("")
# 3) Earliest non-empty message timestamp among these frontends
ear_epoch = earliest_overall.get('earliest_non_empty_timestamp_epoch_s', pd.NA)
ear_str = earliest_overall.get('earliest_non_empty_timestamp_UTC+8', pd.NaT)
lines.append("3) Earliest non-empty message among the three frontends:")
lines.append(f"- epoch_s: {int(ear_epoch) if pd.notna(ear_epoch) else 'N/A'}; UTC+8: {ear_str if pd.notna(ear_str) else 'N/A'}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  non_empty_message_count  earliest_timestamp_epoch_s  earliest_timestamp_UTC+8  latest_timestamp_epoch_s    latest_timestamp_UTC+8
0  frontend-0                      546                  1647865802 2022-03-21 20:30:02+08:00                1647867599 2022-03-21 20:59:59+08:00
1  frontend-1                      406                  1647865801 2022-03-21 20:30:01+08:00                1647867597 2022-03-21 20:59:57+08:00
2  frontend-2                      412                  1647865801 2022-03-21 20:30:01+08:00                1647867598 2022-03-21 20:59:58+08:00,     timestamp           timestamp_UTC+8  message_count
0  1647866253 2022-03-21 20:37:33+08:00              4
1  1647867276 2022-03-21 20:54:36+08:00              3
2  1647865908 2022-03-21 20:31:48+08:00              3
3  1647866610 2022-03-21 20:43:30+08:00              3
4  1647866007 2022-03-21 20:33:27+08:00              3
5  1647866029 2022-03-21 20:33:49+08:00              3
6  1647866003 2022-03-21 20:33:23+08:00              3
7  1647866474 2022-03-21 20:41:14+08:00              3
8  1647866188 2022-03-21 20:36:28+08:00              3
9  1647866243 2022-03-21 20:37:23+08:00              3, earliest_non_empty_timestamp_epoch_s                   1647865801
earliest_non_empty_timestamp_UTC+8      2022-03-21 20:30:01+08:00
dtype: object)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Reuse trace_df from kernel
assert 'trace_df' in globals(), "trace_df not found in kernel."

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df = trace_df.dropna(subset=['value'])

# 1) Global P95 per (cmdb_id, trace_name) using full series
p95_series_trace = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95)
p95_trace_df = p95_series_trace.reset_index().rename(columns={'value': 'p95'})

# 2) Filter trace data to incident window (UTC+8): 2022-03-21 20:30 to 21:00
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 21, 20, 30, 0))
end_dt = tz.localize(datetime(2022, 3, 21, 21, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

window_trace = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# Merge P95 into windowed data
window_trace = window_trace.merge(p95_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# Compute is_anom: value >= p95 (p95 may be NaN; those will be False)
mask_p95_valid = window_trace['p95'].notna()
window_trace['is_anom'] = False
window_trace.loc[mask_p95_valid, 'is_anom'] = window_trace.loc[mask_p95_valid, 'value'] >= window_trace.loc[mask_p95_valid, 'p95']

# Aggregations per (cmdb_id, trace_name)
agg_total_trace = window_trace.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    total_points_in_window=('timestamp', 'size'),
    max_value_in_window=('value', 'max')
).reset_index()

agg_anom_trace = window_trace[window_trace['is_anom']].groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    anomaly_count=('is_anom', 'sum'),
    earliest_anomaly_timestamp=('timestamp', 'min')
).reset_index()

# Merge aggregates and p95
trace_summary = agg_total_trace.merge(p95_trace_df, on=['cmdb_id', 'trace_name'], how='left')
trace_summary = trace_summary.merge(agg_anom_trace, on=['cmdb_id', 'trace_name'], how='left')

# Fill NaNs for combos with no anomalies
trace_summary['anomaly_count'] = trace_summary['anomaly_count'].fillna(0).astype(int)
trace_summary['earliest_anomaly_timestamp'] = trace_summary['earliest_anomaly_timestamp'].where(
    trace_summary['earliest_anomaly_timestamp'].notna(), np.nan
)

# severity_ratio = max_value_in_window / p95 (safe: NaN if p95 is NaN or zero)
trace_summary['severity_ratio'] = np.where(
    (trace_summary['p95'].notna()) & (trace_summary['p95'] != 0),
    trace_summary['max_value_in_window'] / trace_summary['p95'],
    np.nan
)

# 3) Top 20 combos with anomaly_count > 0 ranked by severity_ratio desc
anom_trace_df = trace_summary[trace_summary['anomaly_count'] > 0].copy()

top20_traces = (anom_trace_df
                .sort_values(by=['severity_ratio', 'anomaly_count'], ascending=[False, False], na_position='last')
                .head(20)
                .reset_index(drop=True)
               )

# Select and order required columns for compact display
top20_traces = top20_traces[[
    'cmdb_id',
    'trace_name',
    'p95',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'severity_ratio',
    'total_points_in_window'
]].copy()

# Format earliest_anomaly_timestamp to UTC+8 datetime column for readability (keep epoch too)
if not top20_traces.empty:
    top20_traces['earliest_anomaly_timestamp_UTC+8'] = pd.to_datetime(
        top20_traces['earliest_anomaly_timestamp'].dropna().astype('Int64'), unit='s', utc=True
    ).dt.tz_convert(tz)
else:
    top20_traces['earliest_anomaly_timestamp_UTC+8'] = pd.Series(dtype='datetime64[ns]')

# 4a) total number of distinct combos with anomaly_count > 0
distinct_anomalous_traces = int((anom_trace_df.shape[0]))

# 4b) earliest anomaly timestamp among all trace anomalies
valid_earliest_trace = anom_trace_df['earliest_anomaly_timestamp'].dropna()
if not valid_earliest_trace.empty:
    earliest_anomaly_epoch = int(valid_earliest_trace.min())
    earliest_anomaly_UTC8 = pd.to_datetime(earliest_anomaly_epoch, unit='s', utc=True).tz_convert(tz)
else:
    earliest_anomaly_epoch = pd.NA
    earliest_anomaly_UTC8 = pd.NaT

earliest_anomaly_df = pd.DataFrame({
    'earliest_anomaly_epoch_s': [earliest_anomaly_epoch],
    'earliest_anomaly_UTC+8': [earliest_anomaly_UTC8]
})

# 4c) top 5 cmdb_ids by count of distinct anomalous trace features
top5_cmdb_traces = (anom_trace_df.groupby('cmdb_id', dropna=False)['trace_name']
                    .nunique()
                    .reset_index(name='distinct_anomalous_trace_count')
                    .sort_values('distinct_anomalous_trace_count', ascending=False)
                    .head(5)
                    .reset_index(drop=True)
                   )

# Keep outputs compact: top20_traces (<=20 rows), distinct_anomalous_traces (scalar),
# earliest_anomaly_df (single-row), top5_cmdb_traces (<=5 rows)
top20_traces, distinct_anomalous_traces, earliest_anomaly_df, top5_cmdb_traces

```
Out[5]:
```
```python
# Build a concise plain-English summary from the kernel results:
# Variables used: top20_traces, distinct_anomalous_traces, earliest_anomaly_df, top5_cmdb_traces

import pandas as pd

# Prepare readable top entries (take up to 5 for compactness)
top_n = min(5, len(top20_traces))
rows = []
for i, r in top20_traces.head(top_n).iterrows():
    cmdb = r['cmdb_id']
    trace = r['trace_name']
    p95 = r['p95']
    anom = int(r['anomaly_count'])
    earliest_epoch = r['earliest_anomaly_timestamp']
    earliest_str = str(r['earliest_anomaly_timestamp_UTC+8']) if pd.notna(r['earliest_anomaly_timestamp_UTC+8']) else "N/A"
    maxv = r['max_value_in_window']
    sev = r['severity_ratio']
    pts = int(r['total_points_in_window'])
    rows.append(f"{i+1}) {cmdb} / {trace} — p95={p95}, anomalies={anom}, earliest={earliest_epoch} ({earliest_str}), max={maxv}, severity_ratio={sev:.3f}, points={pts}")

# Scalars
total_anomalous = int(distinct_anomalous_traces)
earliest_epoch_all = int(earliest_anomaly_df.loc[0, 'earliest_anomaly_epoch_s']) if pd.notna(earliest_anomaly_df.loc[0, 'earliest_anomaly_epoch_s']) else "N/A"
earliest_utc8_all = str(earliest_anomaly_df.loc[0, 'earliest_anomaly_UTC+8'])

# Top 5 cmdb_ids by distinct anomalous trace count
top5_lines = []
for _, r in top5_cmdb_traces.iterrows():
    top5_lines.append(f"{r['cmdb_id']} ({int(r['distinct_anomalous_trace_count'])})")

summary_lines = [
    "Trace anomaly summary (window 2022-03-21 20:30–21:00 UTC+8):",
    "",
    f"- Total distinct (cmdb_id, trace_name) combos with anomaly_count>0: {total_anomalous}",
    f"- Earliest anomaly among all traces: epoch {earliest_epoch_all}, UTC+8 {earliest_utc8_all}",
    "",
    f"- Top {top_n} trace combos by severity_ratio (excerpt):"
]
summary_lines += rows
summary_lines += [
    "",
    "- Top 5 cmdb_ids by number of distinct anomalous trace features:",
]
summary_lines += [f"- {s}" for s in top5_lines]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                trace_name       p95  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  severity_ratio  total_points_in_window earliest_anomaly_timestamp_UTC+8
0   checkoutservice-0       trace.from_frontend-2.duration_mean  0.066336              2                  1647866820            59.999959      904.488361                      25        2022-03-21 20:47:00+08:00
1   checkoutservice-0        trace.from_frontend-2.duration_p95  0.066336              2                  1647866820            59.999959      904.488361                      25        2022-03-21 20:47:00+08:00
2          frontend-2  trace.to_checkoutservice-0.duration_mean  0.066336              2                  1647866820            59.999959      904.488361                      25        2022-03-21 20:47:00+08:00
3          frontend-2   trace.to_checkoutservice-0.duration_p95  0.066336              2                  1647866820            59.999959      904.488361                      25        2022-03-21 20:47:00+08:00
4   checkoutservice-0       trace.from_frontend-1.duration_mean  0.084374              2                  1647866640            59.999652      711.117522                      26        2022-03-21 20:44:00+08:00
5   checkoutservice-0        trace.from_frontend-1.duration_p95  0.084374              2                  1647866640            59.999652      711.117522                      26        2022-03-21 20:44:00+08:00
6          frontend-1  trace.to_checkoutservice-0.duration_mean  0.084374              2                  1647866640            59.999652      711.117522                      26        2022-03-21 20:44:00+08:00
7          frontend-1   trace.to_checkoutservice-0.duration_p95  0.084374              2                  1647866640            59.999652      711.117522                      26        2022-03-21 20:44:00+08:00
8       cartservice-1        trace.from_frontend-0.duration_p95  0.000300              3                  1647866880             0.024500       81.666667                      30        2022-03-21 20:48:00+08:00
9          frontend-0       trace.to_cartservice-1.duration_p95  0.000300              3                  1647866880             0.024500       81.666667                      30        2022-03-21 20:48:00+08:00
10  checkoutservice-1       trace.from_frontend-0.duration_mean  0.718941              2                  1647866520            51.820949       72.079597                      25        2022-03-21 20:42:00+08:00
11         frontend-0  trace.to_checkoutservice-1.duration_mean  0.718941              2                  1647866520            51.820949       72.079597                      25        2022-03-21 20:42:00+08:00
12  checkoutservice-1        trace.from_frontend-0.duration_p95  0.755483              2                  1647866520            51.820949       68.593119                      25        2022-03-21 20:42:00+08:00
13         frontend-0   trace.to_checkoutservice-1.duration_p95  0.755483              2                  1647866520            51.820949       68.593119                      25        2022-03-21 20:42:00+08:00
14  checkoutservice-1       trace.from_frontend-2.duration_mean  0.811517              2                  1647866400            25.817599       31.813983                      22        2022-03-21 20:40:00+08:00
15  checkoutservice-1        trace.from_frontend-2.duration_p95  0.811517              2                  1647866400            25.817599       31.813983                      22        2022-03-21 20:40:00+08:00
16         frontend-2  trace.to_checkoutservice-1.duration_mean  0.811517              2                  1647866400            25.817599       31.813983                      22        2022-03-21 20:40:00+08:00
17         frontend-2   trace.to_checkoutservice-1.duration_p95  0.811517              2                  1647866400            25.817599       31.813983                      22        2022-03-21 20:40:00+08:00
18      cartservice-1       trace.from_frontend-0.duration_mean  0.000311              2                  1647866100             0.004455       14.315553                      30        2022-03-21 20:35:00+08:00
19         frontend-0      trace.to_cartservice-1.duration_mean  0.000311              2                  1647866100             0.004455       14.315553                      30        2022-03-21 20:35:00+08:00, 1232,    earliest_anomaly_epoch_s    earliest_anomaly_UTC+8
0                1647865800 2022-03-21 20:30:00+08:00,              cmdb_id  distinct_anomalous_trace_count
0         frontend-1                              92
1  checkoutservice-2                              92
2         frontend-0                              92
3         frontend-2                              92
4  checkoutservice-1                              92)```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing kernel variables if present
assert 'trace_summary' in globals(), "trace_summary not found in kernel. Please compute trace anomalies first."
# metric anomaly summary from earlier metric analysis
metric_anom_df = globals().get('summary_df', None)

# Timezone for formatting
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter trace_summary for anomaly_count > 0
anom_traces = trace_summary[trace_summary['anomaly_count'] > 0].copy()

# 2) Derive downstream service per rules
def extract_downstream(trace_name, row_cmdb):
    # rule 1: look for 'to_<service_token>' and capture up to next '.' or end
    m = re.search(r'to_([^\.]+)', str(trace_name))
    if m:
        return m.group(1)
    # rule 2: else if contains 'from_' then downstream is the row's cmdb_id
    if 'from_' in str(trace_name):
        return row_cmdb
    # rule 3: otherwise skip (return None)
    return None

anom_traces['downstream'] = anom_traces.apply(
    lambda r: extract_downstream(r['trace_name'], r['cmdb_id']), axis=1
)

# Keep only mapped traces
mapped_traces = anom_traces[anom_traces['downstream'].notna()].copy()

# 3) Aggregate by downstream service
agg_by_downstream = mapped_traces.groupby('downstream', dropna=False).agg(
    anomalous_trace_feature_count = ('trace_name', lambda s: s.nunique()),
    distinct_cmdb_trace_combo_count = ('trace_name', lambda s: s.size),  # per mapping this equals number of rows mapped
    earliest_anomaly_epoch_s = ('earliest_anomaly_timestamp', 'min')
).reset_index().rename(columns={'downstream':'downstream_service'})

# Format earliest anomaly to UTC+8
agg_by_downstream['earliest_anomaly_UTC+8'] = pd.to_datetime(
    agg_by_downstream['earliest_anomaly_epoch_s'].dropna().astype('Int64'), unit='s', utc=True
).dt.tz_convert(tz)

# 4) Compute metric anomaly evidence per downstream service:
# If metric_anom_df (summary_df) not available, attempt to recompute would be required;
# here we expect metric_anom_df exists in kernel (from prior steps).
if metric_anom_df is None:
    # fallback: create empty metric evidence counts
    metric_evidence = pd.DataFrame(columns=['cmdb_id','metric_anomalous_combo_count'])
else:
    # count distinct (cmdb_id, kpi_name) combos with anomaly_count>0
    metric_evidence = (metric_anom_df[metric_anom_df['anomaly_count'] > 0]
                       .groupby('cmdb_id', dropna=False)
                       .agg(metric_anomalous_combo_count=('kpi_name', 'nunique'))
                       .reset_index())

# Merge metric evidence into downstream aggregation (match downstream_service to metric cmdb_id)
agg = agg_by_downstream.merge(metric_evidence, left_on='downstream_service', right_on='cmdb_id', how='left')
agg['metric_anomalous_combo_count'] = agg['metric_anomalous_combo_count'].fillna(0).astype(int)
agg = agg.drop(columns=['cmdb_id'])

# 5) Prepare compact top-10 table ranked by anomalous_trace_feature_count desc
top10_downstream = agg.sort_values(
    by=['anomalous_trace_feature_count', 'metric_anomalous_combo_count'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# Keep only requested columns and compact formatting
top10_display = top10_downstream[[
    'downstream_service',
    'anomalous_trace_feature_count',
    'earliest_anomaly_epoch_s',
    'earliest_anomaly_UTC+8',
    'metric_anomalous_combo_count'
]].copy()

# Ensure earliest_anomaly_epoch_s is int where present
top10_display['earliest_anomaly_epoch_s'] = top10_display['earliest_anomaly_epoch_s'].astype('Int64')

# Final compact result
top10_display

```
Out[6]:
```
Here’s a concise summary of downstream services mapped from anomalous traces (rules: prefer "to_<service>" token; otherwise "from_" ⇒ use row cmdb_id). Table shows downstream_service — anomalous_trace_feature_count — earliest anomaly (UTC+8) — metric_anomalous_combo_count:

- productcatalogservice-0 — 44 anomalous trace features — earliest 2022-03-21 20:30:00+08:00 — 116 metric anomalous combos  
- productcatalogservice-1 — 44 — earliest 2022-03-21 20:30:00+08:00 — 116 metric anomalous combos  
- productcatalogservice-2 — 44 — earliest 2022-03-21 20:30:00+08:00 — 116 metric anomalous combos  
- cartservice-1 — 28 — earliest 2022-03-21 20:30:00+08:00 — 96 metric anomalous combos  
- cartservice-0 — 28 — earliest 2022-03-21 20:30:00+08:00 — 92 metric anomalous combos  
- cartservice-2 — 28 — earliest 2022-03-21 20:30:00+08:00 — 88 metric anomalous combos  
- shippingservice-0 — 28 — earliest 2022-03-21 20:30:00+08:00 — 88 metric anomalous combos  
- shippingservice-1 — 28 — earliest 2022-03-21 20:30:00+08:00 — 80 metric anomalous combos  
- shippingservice-2 — 28 — earliest 2022-03-21 20:30:00+08:00 — 80 metric anomalous combos  
- currencyservice-0 — 28 — earliest 2022-03-21 20:30:00+08:00 — 76 metric anomalous combos

Interpretation: productcatalogservice instances show the largest number of distinct anomalous trace features (44 each) with metric anomaly evidence (116 distinct metric combos). Several cartservice and shippingservice pods also show many anomalous trace features (28) and substantial metric anomaly counts, all with earliest anomalies at the start of the incident window (2022-03-21 20:30:00 UTC+8).

The original code execution output of IPython Kernel is also provided below for reference:

downstream_service  anomalous_trace_feature_count  earliest_anomaly_epoch_s    earliest_anomaly_UTC+8  metric_anomalous_combo_count
0  productcatalogservice-0                             44                1647865800 2022-03-21 20:30:00+08:00                           116
1  productcatalogservice-1                             44                1647865800 2022-03-21 20:30:00+08:00                           116
2  productcatalogservice-2                             44                1647865800 2022-03-21 20:30:00+08:00                           116
3            cartservice-1                             28                1647865800 2022-03-21 20:30:00+08:00                            96
4            cartservice-0                             28                1647865800 2022-03-21 20:30:00+08:00                            92
5            cartservice-2                             28                1647865800 2022-03-21 20:30:00+08:00                            88
6        shippingservice-0                             28                1647865800 2022-03-21 20:30:00+08:00                            88
7        shippingservice-1                             28                1647865800 2022-03-21 20:30:00+08:00                            80
8        shippingservice-2                             28                1647865800 2022-03-21 20:30:00+08:00                            80
9        currencyservice-0                             28                1647865800 2022-03-21 20:30:00+08:00                            76```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
import re
from datetime import datetime

# Ensure metric anomaly summary (summary_df) exists; if not, recompute metric anomalies using full metric series.
if 'summary_df' not in globals():
    # Recompute metric anomalies similar to prior steps
    assert 'metric_df' in globals(), "metric_df not found; cannot recompute metric anomalies."
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_df = metric_df.dropna(subset=['value'])
    # Compute global P95 per (cmdb_id, kpi_name)
    p95_series = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95)
    p95_df = p95_series.reset_index().rename(columns={'value': 'p95'})
    # Incident window UTC+8
    tz = pytz.timezone('Asia/Shanghai')
    start_ts = int(tz.localize(datetime(2022,3,21,20,30,0)).timestamp())
    end_ts = int(tz.localize(datetime(2022,3,21,21,0,0)).timestamp())
    window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
    window_df = window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    mask_p95_valid = window_df['p95'].notna()
    window_df['is_anom'] = False
    window_df.loc[mask_p95_valid, 'is_anom'] = window_df.loc[mask_p95_valid, 'value'] >= window_df.loc[mask_p95_valid, 'p95']
    agg_total = window_df.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        total_points_in_window=('timestamp','size'),
        max_value_in_window=('value','max')
    ).reset_index()
    agg_anom_counts = window_df[window_df['is_anom']].groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        anomaly_count=('is_anom','sum'),
        earliest_anomaly_timestamp=('timestamp','min')
    ).reset_index()
    summary_df = agg_total.merge(p95_df, on=['cmdb_id','kpi_name'], how='left').merge(agg_anom_counts, on=['cmdb_id','kpi_name'], how='left')
    summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
    summary_df['severity_ratio'] = np.where((summary_df['p95'].notna()) & (summary_df['p95'] != 0),
                                           summary_df['max_value_in_window'] / summary_df['p95'],
                                           np.nan)

# Build components list: top downstream services previously computed plus node-5 and node-6.
# Use kernel variable top10_display if available, else attempt to use top10_downstream or top10_display fallback.
components_source = None
if 'top10_display' in globals():
    components_source = top10_display['downstream_service'].tolist()
elif 'top10_downstream' in globals():
    components_source = top10_downstream['downstream_service'].tolist()
else:
    # fallback: use most frequent downstream from previous agg if present
    components_source = globals().get('top10_downstream', pd.DataFrame()).get('downstream_service', []).tolist()

# Ensure we have up to 10 downstream services
components = list(dict.fromkeys(components_source))[:10] if components_source else []
# Add node-5 and node-6
components += ['node-5', 'node-6']
# Limit to <=12 rows
components = components[:12]

# Define resource-type bucketing function
def resource_bucket(kpi):
    if pd.isna(kpi):
        return 'other'
    k = str(kpi).lower()
    if ('cpu' in k) or ('throttle' in k) or ('throttled' in k):
        return 'cpu'
    if ('memory' in k) or ('heap' in k):
        return 'memory'
    if ('network' in k) or ('receive' in k) or ('transmit' in k) or ('packet' in k) or ('mb' in k) or ('err' in k):
        return 'network'
    if ('io' in k) or ('read' in k) or ('write' in k) or ('avg_q_sz' in k) or ('disk' in k):
        return 'io'
    if ('file_descriptor' in k) or ('sockets' in k) or ('socket' in k) or ('file descriptor' in k):
        return 'fd'
    return 'other'

# Prepare metric-anomalous combos subset
metric_anoms = summary_df[summary_df['anomaly_count'] > 0].copy()

# For each component, compute requested aggregations
rows = []
for comp in components:
    comp_rows = metric_anoms[metric_anoms['cmdb_id'] == comp]
    if comp_rows.empty:
        earliest_epoch = pd.NA
        combo_count = 0
        max_sev = np.nan
        top_res = ""
    else:
        earliest_vals = comp_rows['earliest_anomaly_timestamp'].dropna()
        earliest_epoch = int(earliest_vals.min()) if not earliest_vals.empty else pd.NA
        combo_count = int(comp_rows.shape[0])  # distinct (cmdb_id,kpi_name) combos already unique per row
        max_sev = float(comp_rows['severity_ratio'].dropna().max()) if comp_rows['severity_ratio'].dropna().size > 0 else np.nan
        # resource-type counts
        buckets = comp_rows['kpi_name'].apply(resource_bucket)
        bucket_counts = buckets.value_counts()
        top5 = bucket_counts.head(5)
        top_res = ", ".join([f"{name}({int(cnt)})" for name, cnt in top5.items()])
    rows.append({
        'component': comp,
        'earliest_metric_anomaly_epoch_s': earliest_epoch,
        'metric_anomalous_combo_count': combo_count,
        'max_metric_severity_ratio': max_sev,
        'top_resource_type_counts': top_res
    })

service_summary = pd.DataFrame(rows)

# Keep compact: ensure ordering same as components list
service_summary['component'] = pd.Categorical(service_summary['component'], categories=components, ordered=True)
service_summary = service_summary.sort_values('component').reset_index(drop=True)

# Display the compact table (<=12 rows)
service_summary

```
Out[7]:
```
Summary (service-/node-centric) of metric-anomaly evidence for the selected downstream services plus node-5 and node-6:

All earliest metric anomaly timestamps below are in UTC+8 (epoch 1647865800 = 2022-03-21 20:30:00+08).

1) productcatalogservice-0  
   - earliest_metric_anomaly: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 116  
   - max_metric_severity_ratio: 1.334  
   - top resource-type buckets (top 5): io(65), network(16), memory(15), cpu(10), other(...)

2) productcatalogservice-1  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 116  
   - max severity_ratio: 1.268  
   - top buckets: io(65), network(16), memory(15), cpu(10), other(...)

3) productcatalogservice-2  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 116  
   - max severity_ratio: 1.430  
   - top buckets: io(65), network(16), memory(15), cpu(10), other(...)

4) cartservice-1  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 96  
   - max severity_ratio: 7.234  
   - top buckets: io(49), memory(15), network(12), cpu(10), other(...)

5) cartservice-0  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 92  
   - max severity_ratio: 7.251  
   - top buckets: io(45), memory(15), network(12), cpu(10), other(...)

6) cartservice-2  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 88  
   - max severity_ratio: 7.771  
   - top buckets: io(39), memory(15), network(14), cpu(10), other(...)

7) shippingservice-0  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 88  
   - max severity_ratio: 13.324  
   - top buckets: io(40), memory(15), network(13), cpu(10), other(...)

8) shippingservice-1  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 80  
   - max severity_ratio: 1.973  
   - top buckets: io(32), memory(15), network(13), cpu(10), other(...)

9) shippingservice-2  
   - earliest: 2022-03-21 20:30:00+08  
   - metric_anomalous_combo_count: 80  
   - max severity_ratio: 2.236  
   - top buckets: io(32), memory(15), network(13), cpu(10), other(...)

10) currencyservice-0  
    - earliest: 2022-03-21 20:30:00+08  
    - metric_anomalous_combo_count: 76  
    - max severity_ratio: 1.500  
    - top buckets: io(28), memory(15), network(13), cpu(10), other(...)

11) node-5  
    - earliest: 2022-03-21 20:30:00+08  
    - metric_anomalous_combo_count: 58  
    - max severity_ratio: 6.620  
    - top buckets: other(32), io(14), network(8), cpu(4)

12) node-6  
    - earliest: 2022-03-21 20:30:00+08  
    - metric_anomalous_combo_count: 58  
    - max severity_ratio: 1.538  
    - top buckets: other(32), io(14), network(8), cpu(4)

Interpretation highlights:
- productcatalogservice instances show the largest number of anomalous metric combos (116 each), dominated by IO-related KPIs. Their max severity ratios are modest (~1.27–1.43).
- cartservice and shippingservice pods also show many anomalous metric combos, with shippingservice-0 and cartservice pods exhibiting higher max severity_ratio values (up to ~13.3 and ~7.8), indicating stronger metric spikes for some KPIs.
- node-5 has notable metric severity (max ~6.62) with many anomalies categorized as "other" and IO.
- All components' earliest metric anomaly timestamps are at the start of the incident window (2022-03-21 20:30:00+08), consistent with an incident commencing at that time.

The original code execution output of IPython Kernel is also provided below for reference:

component  earliest_metric_anomaly_epoch_s  metric_anomalous_combo_count  max_metric_severity_ratio                           top_resource_type_counts
0   productcatalogservice-0                       1647865800                           116                   1.333757  io(65), network(16), memory(15), cpu(10), othe...
1   productcatalogservice-1                       1647865800                           116                   1.267818  io(65), network(16), memory(15), cpu(10), othe...
2   productcatalogservice-2                       1647865800                           116                   1.429544  io(65), network(16), memory(15), cpu(10), othe...
3             cartservice-1                       1647865800                            96                   7.233909  io(49), memory(15), network(12), cpu(10), othe...
4             cartservice-0                       1647865800                            92                   7.250617  io(45), memory(15), network(12), cpu(10), othe...
5             cartservice-2                       1647865800                            88                   7.770970  io(39), memory(15), network(14), cpu(10), othe...
6         shippingservice-0                       1647865800                            88                  13.324228  io(40), memory(15), network(13), cpu(10), othe...
7         shippingservice-1                       1647865800                            80                   1.972567  io(32), memory(15), network(13), cpu(10), othe...
8         shippingservice-2                       1647865800                            80                   2.236480  io(32), memory(15), network(13), cpu(10), othe...
9         currencyservice-0                       1647865800                            76                   1.500000  io(28), memory(15), network(13), cpu(10), othe...
10                   node-5                       1647865800                            58                   6.619718              other(32), io(14), network(8), cpu(4)
11                   node-6                       1647865800                            58                   1.538462              other(32), io(14), network(8), cpu(4)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Ensure metric anomaly summary exists; recompute if absent
if 'summary_df' not in globals():
    # Recompute metric anomalies (global P95 then window) - minimal recompute following prior logic
    assert 'metric_df' in globals(), "metric_df not found to recompute metric anomalies."
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_df = metric_df.dropna(subset=['value'])
    # global P95 per (cmdb_id, kpi_name)
    p95_series = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95)
    p95_df = p95_series.reset_index().rename(columns={'value':'p95'})
    # incident window UTC+8
    tz = pytz.timezone('Asia/Shanghai')
    start_ts = int(tz.localize(datetime(2022,3,21,20,30,0)).timestamp())
    end_ts = int(tz.localize(datetime(2022,3,21,21,0,0)).timestamp())
    window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
    window_df = window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    mask_p95_valid = window_df['p95'].notna()
    window_df['is_anom'] = False
    window_df.loc[mask_p95_valid, 'is_anom'] = window_df.loc[mask_p95_valid, 'value'] >= window_df.loc[mask_p95_valid, 'p95']
    agg_total = window_df.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        total_points_in_window=('timestamp','size'),
        max_value_in_window=('value','max')
    ).reset_index()
    agg_anom_counts = window_df[window_df['is_anom']].groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        anomaly_count=('is_anom','sum'),
        earliest_anomaly_timestamp=('timestamp','min')
    ).reset_index()
    summary_df = agg_total.merge(p95_df, on=['cmdb_id','kpi_name'], how='left').merge(agg_anom_counts, on=['cmdb_id','kpi_name'], how='left')
    summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
    summary_df['severity_ratio'] = np.where((summary_df['p95'].notna()) & (summary_df['p95'] != 0),
                                           summary_df['max_value_in_window'] / summary_df['p95'],
                                           np.nan)

# Use UTC+8 tz for formatting
tz = pytz.timezone('Asia/Shanghai')

# Total distinct anomalous metric combos across all components
metric_anoms = summary_df[summary_df['anomaly_count'] > 0].copy()
total_distinct_anomalous = int(metric_anoms.shape[0])

# Function to aggregate node-specific metrics by searching for 'node-5' / 'node-6' in kpi_name
def node_aggregate(node_token):
    mask = metric_anoms['kpi_name'].astype(str).str.contains(node_token, na=False)
    sel = metric_anoms[mask].copy()
    distinct_count = int(sel.shape[0])
    if distinct_count > 0:
        earliest_vals = sel['earliest_anomaly_timestamp'].dropna()
        earliest_epoch = int(earliest_vals.min()) if not earliest_vals.empty else pd.NA
        max_sev = float(sel['severity_ratio'].dropna().max()) if sel['severity_ratio'].dropna().size > 0 else np.nan
    else:
        earliest_epoch = pd.NA
        max_sev = np.nan
    pct = (distinct_count / total_distinct_anomalous * 100) if total_distinct_anomalous > 0 else 0.0
    return {
        'node_id': node_token,
        'distinct_anomalous_metric_combo_count': distinct_count,
        'earliest_metric_anomaly_epoch_s': earliest_epoch,
        'earliest_metric_anomaly_UTC+8': pd.to_datetime(earliest_epoch, unit='s', utc=True).tz_convert(tz) if pd.notna(earliest_epoch) else pd.NaT,
        'max_metric_severity_ratio': max_sev,
        'percent_of_all_anomalous_metric_combos': f"{pct:.2f}%"
    }

# Build node summary rows for node-5 and node-6
node5_summary = node_aggregate('node-5')
node6_summary = node_aggregate('node-6')

nodes_summary = pd.DataFrame([node5_summary, node6_summary])

# 2) For each node, top up to 10 affected pod cmdb_ids running on that node:
# Attribution: look for metric_anoms rows where kpi_name contains the node token, then group by cmdb_id (pod) and count distinct combos.
def top_pods_for_node(node_token, top_n=10):
    mask = metric_anoms['kpi_name'].astype(str).str.contains(node_token, na=False)
    sel = metric_anoms[mask].copy()
    if sel.empty:
        return pd.DataFrame(columns=['node_id','pod_cmdb_id','anomalous_combo_count'])
    pod_counts = (sel.groupby('cmdb_id', dropna=False)
                  .agg(anomalous_combo_count=('kpi_name','nunique'))
                  .reset_index()
                  .rename(columns={'cmdb_id':'pod_cmdb_id'}))
    pod_counts = pod_counts.sort_values('anomalous_combo_count', ascending=False).head(top_n).reset_index(drop=True)
    pod_counts.insert(0, 'node_id', node_token)
    return pod_counts[['node_id','pod_cmdb_id','anomalous_combo_count']]

pods_node5 = top_pods_for_node('node-5', top_n=10)
pods_node6 = top_pods_for_node('node-6', top_n=10)

# Combine pods lists (limit total to top 20 rows as requested)
pods_combined = pd.concat([pods_node5, pods_node6], ignore_index=True)
pods_combined = pods_combined.reset_index(drop=True)

# Final compact outputs: nodes_summary (2 rows) and pods_combined (up to 20 rows)
nodes_summary, pods_combined

```
Out[8]:
```
Summary (node-focused) derived from the metric-anomaly table:

Node-level summary
- node-5  
  - distinct anomalous metric combos: 2,104 (these are distinct (cmdb_id, kpi_name) combos whose kpi_name references "node-5")  
  - earliest metric anomaly: epoch 1647865800 → 2022-03-21 20:30:00 (UTC+8)  
  - max metric severity_ratio among those combos: 22.021934  
  - percent of all distinct anomalous metric combos: 44.47%

- node-6  
  - distinct anomalous metric combos: 448 (kpi_name references "node-6")  
  - earliest metric anomaly: epoch 1647865800 → 2022-03-21 20:30:00 (UTC+8)  
  - max metric severity_ratio among those combos: 40.706767  
  - percent of all distinct anomalous metric combos: 9.47%

Top affected pods per node (ranked by number of distinct anomalous metric combos)
- For node-5 (top 10 pods, anomalous_combo_count):
  1) adservice-0 — 64  
  2) adservice-1 — 64  
  3) adservice-2 — 64  
  4) checkoutservice-0 — 64  
  5) cartservice-2 — 64  
  6) checkoutservice-1 — 64  
  7) checkoutservice-2 — 64  
  8) currencyservice-0 — 64  
  9) checkoutservice2-0 — 64  
 10) emailservice-0 — 64

- For node-6 (top pods shown, anomalous_combo_count):
  1) adservice2-0 — 64  
  2) cartservice2-0 — 64  
  3) currencyservice2-0 — 64  
  4) frontend2-0 — 64  
  5) paymentservice2-0 — 64  
  6) productcatalogservice2-0 — 64  
  7) recommendationservice2-0 — 64

Key points / interpretation
- Both nodes show their earliest anomalies at the very start of the incident window (20:30:00 UTC+8).  
- node-5 accounts for a large share (~44%) of all distinct anomalous metric combos observed; node-6 contributes ~9.5%.  
- Although node-6 has fewer anomalous combos overall, its maximum severity_ratio is higher (≈40.7) versus node-5 (≈22.0), indicating some very large spikes tied to node-6-related KPIs.  
- Many pods on each node show identical high counts of distinct anomalous metric combos (commonly 64), consistent with a pattern affecting multiple pods on those nodes simultaneously.

(Outputs are counts and timestamps only; no raw messages or time series were included.)

The original code execution output of IPython Kernel is also provided below for reference:

(  node_id  distinct_anomalous_metric_combo_count  earliest_metric_anomaly_epoch_s earliest_metric_anomaly_UTC+8  max_metric_severity_ratio percent_of_all_anomalous_metric_combos
0  node-5                                   2104                       1647865800     2022-03-21 20:30:00+08:00                  22.021934                                 44.47%
1  node-6                                    448                       1647865800     2022-03-21 20:30:00+08:00                  40.706767                                  9.47%,    node_id               pod_cmdb_id  anomalous_combo_count
0   node-5               adservice-0                     64
1   node-5               adservice-1                     64
2   node-5               adservice-2                     64
3   node-5         checkoutservice-0                     64
4   node-5             cartservice-2                     64
5   node-5         checkoutservice-1                     64
6   node-5         checkoutservice-2                     64
7   node-5         currencyservice-0                     64
8   node-5        checkoutservice2-0                     64
9   node-5            emailservice-0                     64
10  node-6              adservice2-0                     64
11  node-6            cartservice2-0                     64
12  node-6        currencyservice2-0                     64
13  node-6               frontend2-0                     64
14  node-6         paymentservice2-0                     64
15  node-6  productcatalogservice2-0                     64
16  node-6  recommendationservice2-0                     64)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric anomaly table 'summary_df' if present; else recompute (should exist from prior steps)
assert 'summary_df' in globals(), "summary_df (metric-anomaly table) not found in kernel."

# Ensure we work only with anomalous combos (anomaly_count > 0)
metric_anoms = summary_df[summary_df['anomaly_count'] > 0].copy()

# UTC+8 timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Pod verification table for the requested pods
pods_list = [
    "productcatalogservice-0", "productcatalogservice-1", "productcatalogservice-2",
    "cartservice-0", "cartservice-1", "cartservice-2",
    "shippingservice-0", "shippingservice-1", "shippingservice-2",
    "frontend-0", "frontend-1", "frontend-2"
]

# Prepare helper columns
metric_anoms['kpi_name_str'] = metric_anoms['kpi_name'].astype(str)

def compute_pod_row(pod):
    sel = metric_anoms[metric_anoms['cmdb_id'] == pod]
    anomalous_combo_count = int(sel.shape[0])
    # node5_kpi_count: kpi_name contains 'node-5'
    node5_kpi_count = int(sel['kpi_name_str'].str.contains('node-5', na=False).sum())
    pct_node5 = f"{(node5_kpi_count / anomalous_combo_count * 100):.2f}%" if anomalous_combo_count > 0 else "0.00%"
    # earliest metric anomaly epoch
    if not sel['earliest_anomaly_timestamp'].dropna().empty:
        earliest_epoch = int(sel['earliest_anomaly_timestamp'].dropna().min())
    else:
        earliest_epoch = pd.NA
    # max severity_ratio
    if not sel['severity_ratio'].dropna().empty:
        max_sev = float(sel['severity_ratio'].dropna().max())
    else:
        max_sev = np.nan
    return {
        'pod_cmdb_id': pod,
        'anomalous_combo_count': anomalous_combo_count,
        'node5_kpi_count': node5_kpi_count,
        'pct_node5': pct_node5,
        'earliest_metric_anomaly_epoch_s': earliest_epoch,
        'earliest_metric_anomaly_UTC+8': pd.to_datetime(earliest_epoch, unit='s', utc=True).tz_convert(tz) if pd.notna(earliest_epoch) else pd.NaT,
        'max_severity_ratio': max_sev
    }

pod_rows = [compute_pod_row(p) for p in pods_list]
pods_verification = pd.DataFrame(pod_rows)

# Ensure epoch column uses Int64 dtype where present
pods_verification['earliest_metric_anomaly_epoch_s'] = pods_verification['earliest_metric_anomaly_epoch_s'].astype('Int64')

# Select and order columns compactly
pods_verification = pods_verification[[
    'pod_cmdb_id',
    'anomalous_combo_count',
    'node5_kpi_count',
    'pct_node5',
    'earliest_metric_anomaly_epoch_s',
    'earliest_metric_anomaly_UTC+8',
    'max_severity_ratio'
]]

# 2) Node IO counts for node-5 and node-6
nodes = ['node-5', 'node-6']

def node_io_counts(node_token):
    sel = metric_anoms[metric_anoms['kpi_name'].astype(str).str.contains(node_token, na=False)].copy()
    # avg_q_sz_count
    avg_q_sz_count = int(sel['kpi_name'].astype(str).str.contains('avg_q_sz', case=False, na=False).sum())
    # disk/read/write count (any of these substrings)
    disk_read_write_mask = sel['kpi_name'].astype(str).str.contains('disk|read|write', case=False, na=False)
    disk_read_write_count = int(disk_read_write_mask.sum())
    return {'node_id': node_token, 'avg_q_sz_count': avg_q_sz_count, 'disk_read_write_count': disk_read_write_count}

node_io_list = [node_io_counts(n) for n in nodes]
node_io_counts = pd.DataFrame(node_io_list)

# Final compact outputs (DataFrames)
pods_verification, node_io_counts

```
Out[9]:
```
Here’s a concise verification summary derived from the metric-anomaly table.

1) Per-pod verification (one row each)
- productcatalogservice-0 — anomalous_combo_count: 116; node5_kpi_count: 64 (55.17%); earliest anomaly: 1647865800 → 2022-03-21 20:30:00+08; max_severity_ratio: 1.3338  
- productcatalogservice-1 — 116; node5_kpi_count: 64 (55.17%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 1.2678  
- productcatalogservice-2 — 116; node5_kpi_count: 64 (55.17%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 1.4295  
- cartservice-0 — 92; node5_kpi_count: 60 (65.22%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 7.2506  
- cartservice-1 — 96; node5_kpi_count: 60 (62.50%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 7.2339  
- cartservice-2 — 88; node5_kpi_count: 64 (72.73%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 7.7710  
- shippingservice-0 — 88; node5_kpi_count: 64 (72.73%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 13.3242  
- shippingservice-1 — 80; node5_kpi_count: 64 (80.00%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 1.9726  
- shippingservice-2 — 80; node5_kpi_count: 64 (80.00%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 2.2365  
- frontend-0 — 180; node5_kpi_count: 64 (35.56%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 1.1940  
- frontend-1 — 180; node5_kpi_count: 64 (35.56%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 112.7052 (very large spike)  
- frontend-2 — 180; node5_kpi_count: 64 (35.56%); earliest: 2022-03-21 20:30:00+08; max_severity_ratio: 2.0000

Notes on the pod table:
- All pods show earliest metric anomalies at the incident start (2022-03-21 20:30:00+08).
- Many pods have a substantial fraction of their anomalous KPIs referencing node-5 (node5_kpi_count and pct_node5 are often high, e.g., 55–80%).
- Notable high severity ratios: frontend-1 is extreme (~112.7); shippingservice-0 (~13.3); cartservice pods (~7.2–7.8).

2) Node IO-related counts (node-5 and node-6)
- node-5 — avg_q_sz_count: 0; disk_read_write_count: 454  
- node-6 — avg_q_sz_count: 0; disk_read_write_count: 98

Interpretation:
- There are many metric anomalies tied to disk/read/write related KPIs on node-5 (454) vs node-6 (98), suggesting heavier I/O-related anomalies on node-5.
- No anomalies matching 'avg_q_sz' were found for either node.
- Combined with the pod breakdown (many pod KPIs referencing node-5), this strengthens evidence that node-5 is heavily implicated in the incident (both many affected pod KPIs and many disk/read/write metric anomalies).

The original code execution output of IPython Kernel is also provided below for reference:

(                pod_cmdb_id  anomalous_combo_count  node5_kpi_count pct_node5  earliest_metric_anomaly_epoch_s earliest_metric_anomaly_UTC+8  max_severity_ratio
0   productcatalogservice-0                    116               64    55.17%                       1647865800     2022-03-21 20:30:00+08:00            1.333757
1   productcatalogservice-1                    116               64    55.17%                       1647865800     2022-03-21 20:30:00+08:00            1.267818
2   productcatalogservice-2                    116               64    55.17%                       1647865800     2022-03-21 20:30:00+08:00            1.429544
3             cartservice-0                     92               60    65.22%                       1647865800     2022-03-21 20:30:00+08:00            7.250617
4             cartservice-1                     96               60    62.50%                       1647865800     2022-03-21 20:30:00+08:00            7.233909
5             cartservice-2                     88               64    72.73%                       1647865800     2022-03-21 20:30:00+08:00            7.770970
6         shippingservice-0                     88               64    72.73%                       1647865800     2022-03-21 20:30:00+08:00           13.324228
7         shippingservice-1                     80               64    80.00%                       1647865800     2022-03-21 20:30:00+08:00            1.972567
8         shippingservice-2                     80               64    80.00%                       1647865800     2022-03-21 20:30:00+08:00            2.236480
9                frontend-0                    180               64    35.56%                       1647865800     2022-03-21 20:30:00+08:00            1.193997
10               frontend-1                    180               64    35.56%                       1647865800     2022-03-21 20:30:00+08:00          112.705172
11               frontend-2                    180               64    35.56%                       1647865800     2022-03-21 20:30:00+08:00            2.000000,   node_id  avg_q_sz_count  disk_read_write_count
0  node-5               0                    454
1  node-6               0                     98)```
```